In [1]:
import requests
import pandas as pd
import time

#### 1. Collect roll calls data and legislators info

##### 1.1 Fetch all documents for the specified session

In [2]:
def get_documents(session_number: int):
    """
    Obtain all documents for the specified session of the General Court
    """
    # Returns all of documents publicly available by the General Court
    url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/"
    print(f"\n Fetching documents from General Court {session_number} ...")

    response = requests.get(url)
    response.raise_for_status()
    docs = response.json()
    print(f"Successfully retrieved {len(docs)} documents from General Court {session_number}.")
    
    return docs


In [3]:
#docs_193 = get_documents(193)

#for i, doc in enumerate(docs_193[:3]):
#   print(f"\n--- Document {i+1} preview ---")
#    print(doc)


##### 1.2 Extract information from documents

In [4]:
import pandas as pd

def extract_bill_info(docs_json, session_number):
    """
    Extract key information from the documents JSON of the Legislature API 
    and convert it into DataFrame format.
    
    Return:
        pd.DataFrame: 
        containing billnumbers, docketnumbers, title, branch, and IsDocketBookOnly
    """
    bill_list = []

    for doc in docs_json:
        bill_no = doc.get("BillNumber")
        docket_no = doc.get("DocketNumber")
        title = doc.get("Title")
        is_book_only = doc.get("IsDocketBookOnly")

        bill_list.append({
            "BillNumber": bill_no if bill_no else None,
            "docketNumber": docket_no if docket_no else None,
            "title": title if title else None,
            "branch": (
                "Senate" if bill_no and bill_no.startswith("S") else
                "House" if bill_no and bill_no.startswith("H") else None
            ),
            "isDocketBookOnly": is_book_only
        })

    df = pd.DataFrame(bill_list)
    print(f"Extracted {len(df)} entries from documents for General Court {session_number if session_number else ''}...")
    #print("Preview of first 5 rows:")
    #print(df.head())
    
    #print("\nMissing value summary:")
    #print(df.isna().sum())

    #print("\n Count by branch:")
    #print(df['branch'].value_counts(dropna=False))

    return df


#df_193 = extract_bill_info(docs_193, 193)


##### 1.3 Filter bills containing rollcall records

Filter records where billnumber is empty, then call the API using billnumber. This API will return rollcall data 
(normally, the API only returns Senate rollcall number records, but for certain sessions, the returned rollcall may list House members).

In [5]:

def get_rollcall_status(df_bills, session_number=None):
    """
    Scan each bill to determine whether a roll call vote record exists.
    """

    # Filter bills with billnumbers：billnumber is not null
    bill_numbers = df_bills[df_bills['BillNumber'].notna()]['BillNumber'].tolist()
    total = len(bill_numbers)
    rollcall_records = []

    print(f"\nStarting roll call scan for General Court {session_number if session_number else ''}")
    print(f"Total bills to scan: {total}\n")

    for index, bill in enumerate(bill_numbers):
        url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Documents/{bill}/RollCalls"
        

        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # Extract roll call number
            if isinstance(data, list) and len(data) > 0:
                roll_nums = [item.get("RollCallNumber") for item in data if "RollCallNumber" in item]
            else:
                roll_nums = []

            rollcall_records.append({
                "BillNumber": bill,
                "rollCallNumbers": roll_nums,
                "hasRollCall": len(roll_nums) > 0
            })

        except Exception as e:
            rollcall_records.append({
                "BillNumber": bill,
                "rollCallNumbers": None,
                "hasRollCall": None
            })
            #print(f"Error fetching {bill}: {e}")

        if (index + 1) % 500 == 0 or (index + 1) == total:
            print(f"📍 Progress: {index + 1}/{total} bills processed...")

        #time.sleep(delay)

    df_rollcall = pd.DataFrame(rollcall_records)

    print("\nRoll Call Scan Summary:")
    try:
        print(df_rollcall['hasRollCall'].value_counts(dropna=False))
    except:
        print(f"0 Roll Calls in session {session_number}")
    print("\nExample preview:")
    print(df_rollcall.head())
    print(df_rollcall.shape)

    return df_rollcall

#df_bills_193 = get_rollcall_status(df_193, session_number=193)


All roll call records for each bill have been obtained. For bills without a roll call record, the `hasRollCall` field will be marked as false.

##### 1.4 Fetch the details of each vote by every member

This step will filter out bills containing rollcall votes, then iterate through the API to retrieve their corresponding rollcall numbers,
obtaining the detailed voting lists for each rollcall vote.

In [6]:
def fetch_bill_vote_details(df_roll_status, session_number=None, delay=0.2):
    """
    Retrieve all voting details for legislators who participated in roll call votes 
    from the existing bill roll call status table.
    """
    # Select only bills that have had roll call votes.
    try:
        df_with_rolls = df_roll_status[df_roll_status['hasRollCall'] == True].copy()
        total_bills = len(df_with_rolls)
        print(f"\nFetching vote details for {total_bills} bills with roll calls "
                f"in General Court {session_number if session_number else ''}...\n")
    except:
        print(f"\nFetching vote details for 0 bills with roll calls "
                f"in General Court {session_number if session_number else ''}...\n")
    all_vote_records = []
    try:
        iterations = df_with_rolls.iterrows()
    except:
        iterations = (0,0)
    
    # Iterate through each bill with a roll call
    if iterations != (0,0):
        for idx, row in iterations:
            bill_no = row['BillNumber']
            roll_calls = row['rollCallNumbers'] if isinstance(row['rollCallNumbers'], list) else []
            if not roll_calls:
                continue

            for roll_num in roll_calls:
                url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/Branches/Senate/RollCalls/{roll_num}"
                try:
                    resp = requests.get(url, timeout=8)
                    resp.raise_for_status()
                    data = resp.json()

                    # Extract the list of votes
                    yea_list = data.get("Yeas", [])
                    nay_list = data.get("Nays", [])
                    absent_list = data.get("Absent", [])

                    for vote_list, label in [(yea_list, "Yea"), (nay_list, "Nay"), (absent_list, "Absent")]:
                        for member in vote_list:
                            all_vote_records.append({
                                "GeneralCourtNumber": data.get("GeneralCourtNumber"),
                                "Branch": data.get("Branch"),
                                "RollCallNumber": roll_num,
                                "QuestionMotion": data.get("QuestionMotion"),  
                                "BillNumber": bill_no,
                                "MemberCode": member.get("MemberCode"),
                                "Vote": label,
                                "MemberUrl": member.get("Details")
                            })

                    #print(f"RollCall {roll_num} from {bill_no}: "
                            #f"{len(yea_list)}Y / {len(nay_list)}N / {len(absent_list)}A")

                except Exception as e:
                    print(f"Error fetching {bill_no} - RollCall {roll_num}: {e}")

                time.sleep(delay)  

    df_votes = pd.DataFrame(all_vote_records)
    print(f"\n Completed. Total vote records fetched: {len(df_votes)}")
    return df_votes


#df_vote_details_193 = fetch_bill_vote_details(df_bills_193, session_number=193, delay=0.1)

In [7]:
#df_vote_details_193.head()

In [8]:
#df_with_rolls = df_bills_193[df_bills_193['hasRollCall'] == True].copy()
#df_with_rolls

##### Fetch legislators info for the specified session

Identify all members through the obtained voting details table and obtain their personal information.

In [9]:
def fetch_legislators_from_votes(df_vote_details, session_number, delay=0.2):
    """
    Extract all MemberCodes from the vote details table,
    and call the API to retrieve detailed information on all legislators for that session.
    """

    # Extract all unique MemberCodes
    try:
        unique_codes = df_vote_details["MemberCode"].dropna().unique().tolist()
        total = len(unique_codes)
        print(f"\nFound {total} unique legislators in General Court {session_number}.\n")
    except:
        print(f"\nFound 0 unique legislators in General Court {session_number}.\n")
        unique_codes = -1

    records = []
    if unique_codes != -1:
        # Iterate through each legislator and request data
        for idx, code in enumerate(unique_codes, start=1):
            url = f"https://malegislature.gov/api/GeneralCourts/{session_number}/LegislativeMembers/{code}"
            try:
                resp = requests.get(url, timeout=20)
                resp.raise_for_status()
                data = resp.json()

                record = {
                    "MemberCode": data.get("MemberCode"),
                    "Name": data.get("Name"),
                    "Branch": data.get("Branch"),
                    "Party": data.get("Party"),
                    "District": data.get("District"),
                    "LeadershipPosition": data.get("LeadershipPosition"),
                    "EmailAddress": data.get("EmailAddress"),
                    "PhoneNumber": data.get("PhoneNumber"),
                    "RoomNumber": data.get("RoomNumber"),
                }
                records.append(record)
                #print(f"{idx}/{total} {code} — {record['Name']}")

            except Exception as e:
                print(f"Error fetching {code}: {e}")
                continue

            time.sleep(delay)

    df_legislators = pd.DataFrame(records)
    print(f"\nRetrieved {len(df_legislators)} legislators for General Court {session_number}.")
    
    return df_legislators


In [10]:
#df_legislators_193 = fetch_legislators_from_votes(df_vote_details_193, session_number=193, delay=1)

#df_legislators_193.head()


In [11]:
def build_session_data(session_number, delay_rollcall=0.1, delay_vote=0.1, delay_legislator=0.2):
    """
    Enter the General Court session number and execute the complete data extraction process step by step.
    Return the voting details table and member information table for that session.
    """

    print(f"\nBuilding data for General Court {session_number} ...")

    # Step 1: extract documents
    docs_json = get_documents(session_number)
    df_bills = extract_bill_info(docs_json, session_number)

    # Step 2: extract roll call status
    df_roll_status = get_rollcall_status(df_bills, session_number=session_number)

    # Step 3: extract roll call vote details
    df_vote_details = fetch_bill_vote_details(
        df_roll_status,
        session_number=session_number,
        delay=delay_vote
    )

    # Step 4: extract legislators info
    df_legislators = fetch_legislators_from_votes(
        df_vote_details,
        session_number=session_number,
        delay=delay_legislator
    )

    print(f"Completed session {session_number}.")
    print(f"   Votes: {len(df_vote_details)} records, Legislators: {len(df_legislators)} entries.\n")

    return df_vote_details, df_legislators


In [12]:
# df_vote_details_194, df_legislators_194 = build_session_data(session_number=194)

In [13]:
# df_vote_details_193, df_legislators_193 = build_session_data(session_number=193)
#

In [14]:
# df_vote_details_192, df_legislators_192 = build_session_data(session_number=192)

In [15]:
# df_vote_details_191, df_legislators_191 = build_session_data(session_number=191)

In [16]:
# df_vote_details_190, df_legislators_190 = build_session_data(session_number=190)

In [17]:
# df_vote_details_189, df_legislators_189 = build_session_data(session_number=189)

In [18]:
# df_vote_details_188, df_legislators_188 = build_session_data(session_number=188)

In [19]:
# df_vote_details_187, df_legislators_187 = build_session_data(session_number=187)

In [20]:
# df_vote_details_186, df_legislators_186 = build_session_data(session_number=186)

In [21]:
# df_vote_details_183, df_legislators_183 = build_session_data(session_number=183)

##### 1.5 Collect the committe information to use as topic categorization

In [22]:
#Defining functions.

def fetch_all_committees(GENERAL_COURT):
    """Fetch list of all committees"""
    url = f"https://malegislature.gov/api/GeneralCourts/{GENERAL_COURT}/Committees"
    print(f"Fetching committees from: {url}\n")
    
    try:
        response = requests.get(url, headers={'Accept': 'application/json'})
        response.raise_for_status()
        committees = response.json()
        print(f"Found {len(committees)} committees\n")
        return committees
        
    except Exception as e:
        print(f"Error fetching committees: {e}")
        return []


def fetch_committee_details(committee_code,GENERAL_COURT):
    """Fetch detailed information for a specific committee"""
    url = f"https://malegislature.gov/api/GeneralCourts/{GENERAL_COURT}/Committees/{committee_code}"
    
    try:
        response = requests.get(url, headers={'Accept': 'application/json'})
        response.raise_for_status()
        return response.json()
        
    except Exception as e:
        print(f"Error fetching {committee_code}: {e}")
        return None


def get_bill_type(bill_number):
    """Extract bill type from bill number prefix"""
    if not bill_number:
        return "Unknown"
    
    prefix = ''.join([c for c in bill_number if c.isalpha()])
    
    type_map = {
        'H': 'House Bill',
        'S': 'Senate Bill',
        'HR': 'House Resolution',
        'SR': 'Senate Resolution',
        'HD': 'House Document',
        'SD': 'Senate Document'
    }
    
    return type_map.get(prefix, f'{prefix} Document')


def extract_committee_info(committee):
    """Extract relevant fields from committee data including bills"""
    
    # Get chairperson - check both House and Senate
    chairperson = ''
    if committee.get('SenateChairperson') and committee['SenateChairperson'].get('MemberCode'):
        chairperson = committee['SenateChairperson']['MemberCode']
    elif committee.get('HouseChairperson') and committee['HouseChairperson'].get('MemberCode'):
        chairperson = committee['HouseChairperson']['MemberCode']
    
    # Base committee info
    base_info = {
        'Branch': committee.get('Branch', ''),
        'CommitteeCode': committee.get('CommitteeCode', ''),
        'Chairperson': chairperson,
        'CommitteeName': committee.get('ShortName', '')
    }
    
    # Get bills from committee
    bills = []
    
    # Documents before committee
    docs_before = committee.get('DocumentsBeforeCommittee', [])
    if docs_before:
        for doc in docs_before:
            bill_number = doc.get('DocumentNumber') or doc.get('BillNumber', '')
            if bill_number:
                bills.append({
                    **base_info,
                    'BillNumber': bill_number,
                    'BillType': get_bill_type(bill_number),
                    'Status': 'Before Committee'
                })
    
    # Reported out documents
    reported_out = committee.get('ReportedOutDocuments', [])
    if reported_out:
        for doc in reported_out:
            bill_number = doc.get('DocumentNumber') or doc.get('BillNumber', '')
            if bill_number:
                bills.append({
                    **base_info,
                    'BillNumber': bill_number,
                    'BillType': get_bill_type(bill_number),
                    'Status': 'Reported Out'
                })
    
    # If no bills found, return committee info without bills
    if not bills:
        return [{
            **base_info,
            'BillNumber': '',
            'BillType': '',
            'Status': ''
        }]
    
    return bills



In [23]:
def committeeDataCollection(GENERAL_COURT = 194):
    """ This now fully collects the committee data for a general court for use in the future if needed."""
    GENERAL_COURT = str(GENERAL_COURT)

    # Fetch all committees
    committees = fetch_all_committees(GENERAL_COURT)

    if not committees:
        print("No committees found!")
        exit()

    # Extract committee codes
    committee_codes = [c.get('CommitteeCode') for c in committees if c.get('CommitteeCode')]

    # Fetch details for each committee
    all_details = []
        
    for i, code in enumerate(committee_codes, 1):
        #print(f"[{i}/{len(committee_codes)}] Fetching {code}...", end=" ")
        
        details = fetch_committee_details(code, GENERAL_COURT)
        
        if details:
            committee_rows = extract_committee_info(details)
            all_details.extend(committee_rows)  # extend instead of append since we get a list
            #print(f"✓ ({len(committee_rows)} bills)")
        else:
            print(f"✗ at {code}")
        
        # Be respectful to the API
        time.sleep(0.3)

    # Create DataFrame
    df = pd.DataFrame(all_details)
    try:
        df.dropna(subset=['BillNumber'], axis=0, inplace=True)
    except:
        print("No bills in committee found")    
    #df.rename({"ShortName": "CommitteeName"}, inplace=True)
    df.reset_index(inplace=True, drop=True) #Just resetting index so it doesn't start at random numbers
    # Reorder columns
    df = df[['Branch', 'CommitteeCode', 'Chairperson', 'CommitteeName', 'BillNumber', 'BillType', 'Status']]

    #Big text block saying steps. Can be deleted at some point when everything is done.
    print("=" * 80)
    print("MA Legislature Committee Data Collection")
    print("=" * 80)
    print()
    
    print("\n" + "=" * 80)
    print(f"Successfully fetched {len(all_details)} committee-bill records")
    print("=" * 80)
    # Print DataFrame
    # print("COMMITTEE DATA:")
    # print("=" * 80)
    # print(df.head())
    # print("\n")

    # Show statistics
    # print("SUMMARY:")
    # print("=" * 80)
    # print(f"Total Records: {len(df)}")
    # print(f"Unique Committees: {df['CommitteeCode'].nunique()}")
    # print(f"Total Bills: {df[df['BillNumber'] != '']['BillNumber'].nunique()}")
    # print(f"\nBy Branch:")
    # print(df['Branch'].value_counts().to_string())
    # print(f"\nBy Bill Type:")
    bill_types = df[df['BillType'] != '']['BillType'].value_counts()
    if not bill_types.empty:
        print(bill_types.to_string())
    else:
        print("No bills found")
    # Save to CSV
    #output_file = f"ma_legislature_committees_{GENERAL_COURT}.csv"
    #df.to_csv(output_file, index=False)
    #print(f"\n✓ Saved to: {output_file}")
    return df


In [24]:
def addCommitteeName(df, general_court):
    """ Adds the committees as a column to the dataframe.
    cdf is the committee data while df is the bill data."""
    df["CommitteeName"] = ""
    try:
        df["BillNumber"]
    except:
        print("no Bills")
        return df
    cdf = committeeDataCollection(general_court)    
    for i in range(df["BillNumber"].size):
        try:
            df.loc[i, "CommitteeName"] = cdf["CommitteeName"][(cdf["BillNumber"] == df["BillNumber"][i]).idxmax()]
        except:
            ...
    return df
        
    

In [25]:
#Change the range on which we work to get the data.
# Range can go above whatever the real highest court is as it checks for that quite quickly.
timeline = range(183,220)

In [26]:
#df_vote_details_183, df_legislators_183
votes = {}
legis  = {}
for x in timeline:
    # Skip past any that don't have legislative members as then they are empty. 
    # this request is used as it is faster than the documents request while testing if we have gone past the current date..
    try:
        url = f"https://malegislature.gov/api/GeneralCourts/{x}/LegislativeMembers/"
        response = requests.get(url)
        response.raise_for_status()
    except:
        continue
    votes["df_vote_details_{0}".format(x)], legis["df_legislators_{0}".format(x)] = build_session_data(session_number=x)
    


Building data for General Court 183 ...

 Fetching documents from General Court 183 ...
Successfully retrieved 9 documents from General Court 183.
Extracted 9 entries from documents for General Court 183...

Starting roll call scan for General Court 183
Total bills to scan: 0


Roll Call Scan Summary:
0 Roll Calls in session 183

Example preview:
Empty DataFrame
Columns: []
Index: []
(0, 0)

Fetching vote details for 0 bills with roll calls in General Court 183...


 Completed. Total vote records fetched: 0

Found 0 unique legislators in General Court 183.


Retrieved 0 legislators for General Court 183.
Completed session 183.
   Votes: 0 records, Legislators: 0 entries.


Building data for General Court 184 ...

 Fetching documents from General Court 184 ...
Successfully retrieved 2 documents from General Court 184.
Extracted 2 entries from documents for General Court 184...

Starting roll call scan for General Court 184
Total bills to scan: 1

📍 Progress: 1/1 bills processed...

Rol

In [37]:

for x in timeline:
    try:
        url = f"https://malegislature.gov/api/GeneralCourts/{x}/LegislativeMembers/"
        response = requests.get(url)
        response.raise_for_status()
    except:
        continue
    votes["df_vote_details_{0}".format(x)] = addCommitteeName(votes["df_vote_details_{0}".format(x)], x)

no Bills
no Bills
no Bills
Fetching committees from: https://malegislature.gov/api/GeneralCourts/186/Committees

Found 43 committees

MA Legislature Committee Data Collection


Successfully fetched 43 committee-bill records
No bills found
no Bills
Fetching committees from: https://malegislature.gov/api/GeneralCourts/188/Committees

Found 44 committees

MA Legislature Committee Data Collection


Successfully fetched 11126 committee-bill records
BillType
House Bill     7472
Senate Bill    3645
SI Document       1
Fetching committees from: https://malegislature.gov/api/GeneralCourts/189/Committees

Found 51 committees

MA Legislature Committee Data Collection


Successfully fetched 11959 committee-bill records
BillType
House Bill     8090
Senate Bill    3856
Fetching committees from: https://malegislature.gov/api/GeneralCourts/190/Committees

Found 57 committees

MA Legislature Committee Data Collection


Successfully fetched 12508 committee-bill records
BillType
House Bill     8398
Senat

#### 2. Merge datasets on full session


In [27]:
def merge_legislator_votes(vote_file, legislator_file, output_file=None):
    """
    Merge the legislator info sheet with the voting details sheet into a single comprehensive dataset.
    """
    #  load input
    if isinstance(vote_file, str):
        votes = pd.read_csv(vote_file)
    else:
        votes = vote_file.copy()

    if isinstance(legislator_file, str):
        members = pd.read_csv(legislator_file)
    else:
        members = legislator_file.copy()

    print(f"Loaded {len(votes)} vote records and {len(members)} legislators.")

    # Remove duplicate branch to avoid conflicts
    votes = votes.drop(columns=["Branch"], errors="ignore")
    try:
        merged = pd.merge(members, votes, on="MemberCode", how="left")
    except:
        return votes

    columns = [
        "GeneralCourtNumber", "MemberCode", "Name", "Branch", 
        "BillNumber", "RollCallNumber", "QuestionMotion", "Vote", "Party", "District",
        "LeadershipPosition", "EmailAddress", "PhoneNumber", "RoomNumber", "MemberUrl", "CommitteeName"
    ]
    merged = merged[[c for c in columns if c in merged.columns]]

    # Sorted by Member and Voting Order
    merged["order"] = merged.index
    merged = merged.sort_values(by=["order", "RollCallNumber"], ascending=[True, True])
    merged = merged.drop(columns="order")

    print(f"Merged {len(merged)} rows.")
    return merged



In [38]:
merged_dfs = {}
for i in timeline:
    try:
        legis["df_legislators_{0}".format(i)]["MemberCode"]
    except:
        continue
    merged_dfs[i] = merge_legislator_votes(votes["df_vote_details_{0}".format(i)], legis["df_legislators_{0}".format(i)])

Loaded 1010 vote records and 40 legislators.
Merged 1010 rows.
Loaded 18458 vote records and 42 legislators.
Merged 18280 rows.
Loaded 25739 vote records and 43 legislators.
Merged 25739 rows.
Loaded 21299 vote records and 46 legislators.
Merged 21299 rows.
Loaded 19875 vote records and 192 legislators.
Merged 18387 rows.
Loaded 10217 vote records and 201 legislators.
Merged 10217 rows.
Loaded 9628 vote records and 41 legislators.
Merged 9628 rows.
Loaded 4469 vote records and 40 legislators.
Merged 4469 rows.


In [39]:
merged_dfs[193].head()

,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl,CommitteeName
0,193,PWM0,Paul W. Mark,Senate,H2095,126,None,Yea,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...,"Steering, Policy and Scheduling (H)"
1,193,PWM0,Paul W. Mark,Senate,S2251,58,None,Yea,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...,Ways and Means (H)
2,193,PWM0,Paul W. Mark,Senate,H2049,247,None,Yea,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...,"Steering, Policy and Scheduling (H)"
3,193,PWM0,Paul W. Mark,Senate,S17,2,Question on adoption of the amendment 1,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...,Post Audit and Oversight (H)
4,193,PWM0,Paul W. Mark,Senate,S17,3,Question on adoption of the amendment 2,Nay,Democrat,"Berkshire, Hampden, Franklin and Hampshire",None,Paul.Mark@masenate.gov,6177221625,504,http://malegislature.gov/api/GeneralCourts/193...,Post Audit and Oversight (H)


#### Individual Courts

##### - 194 merged

In [29]:
merged_df_194 = merge_legislator_votes(df_vote_details_194, df_legislators_194)
merged_df_194.head()

NameError: name 'df_vote_details_194' is not defined

In [ ]:
merged_df_194.to_csv("194_merged_legislator_votes.csv", index=False)

##### 193

In [ ]:
merged_df_193 = merge_legislator_votes(df_vote_details_193, df_legislators_193)
merged_df_193.head()

##### 192 Genearl Court

In [ ]:
merged_df_192 = merge_legislator_votes(df_vote_details_192, df_legislators_192)
merged_df_192.head()

Loaded 10217 vote records and 200 legislators.
Merged 10216 rows.


,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,192,LME0,Lydia Edwards,Senate,S1357,156,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
1,192,LME0,Lydia Edwards,Senate,H851,149,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
2,192,LME0,Lydia Edwards,Senate,H2193,120,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
3,192,LME0,Lydia Edwards,Senate,H2294,229,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
4,192,LME0,Lydia Edwards,Senate,H4108,136,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...


In [ ]:
len(merged_df_192["MemberCode"].unique())

200

In [ ]:
vote_counts = (
    merged_df_192
    .groupby(['MemberCode','Name'])
    .size()
    .reset_index(name='VoteCount')
    .sort_values('VoteCount', ascending=False)
)

vote_counts.head(45)


,MemberCode,Name,VoteCount
154,PRF0,Paul R. Feeney,257
129,MRP0,Marc R. Pacheco,257
21,BRF0,Barry R. Finegold,257
153,PMO,Patrick M. O'Connor,257
124,MJR0,Michael J. Rodrigues,257
73,JCV0,John C. Velis,257
54,D_D0,Diana DiZoglio,257
127,MOM0,Michael O. Moore,257
76,JFK0,John F. Keenan,257
158,RCF0,Ryan C. Fattman,257


In [ ]:
bad_codes = vote_counts[vote_counts["VoteCount"] == 1]["MemberCode"].tolist()
#bad_codes[:10]


In [ ]:
bad_rows = merged_df_192[
    merged_df_192["MemberCode"].isin(bad_codes)
].copy()

#bad_rows


In [ ]:
#print(bad_rows['RollCallNumber'].unique())

In [ ]:
merged_df_192_clean = merged_df_192[
    ~merged_df_192["MemberCode"].isin(bad_codes)
].copy()
merged_df_192_clean.head()

,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,192,LME0,Lydia Edwards,Senate,S1357,156,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
1,192,LME0,Lydia Edwards,Senate,H851,149,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
2,192,LME0,Lydia Edwards,Senate,H2193,120,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
3,192,LME0,Lydia Edwards,Senate,H2294,229,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...
4,192,LME0,Lydia Edwards,Senate,H4108,136,None,Yea,Democrat,First Suffolk and Middlesex,None,Lydia.Edwards@masenate.gov,6177221634,520,http://malegislature.gov/api/GeneralCourts/192...


##### 191

In [ ]:
merged_df_191 = merge_legislator_votes(df_vote_details_191, df_legislators_191)
merged_df_191.head()

Loaded 19875 vote records and 189 legislators.
Merged 19771 rows.


,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,191,DAT0,Dean A. Tran,Senate,S63,269,None,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
1,191,DAT0,Dean A. Tran,Senate,H1771,81,None,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
2,191,DAT0,Dean A. Tran,Senate,H86,25,None,Nay,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
3,191,DAT0,Dean A. Tran,Senate,H86,67,Question on adoption of the amendment #1,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
4,191,DAT0,Dean A. Tran,Senate,H86,68,Question on adoption of the amendment #3,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...


In [ ]:
vote_counts = (
    merged_df_191
    .groupby(['MemberCode','Name'])
    .size()
    .reset_index(name='VoteCount')
    .sort_values('VoteCount', ascending=False)
)
vote_counts.head(45)

,MemberCode,Name,VoteCount
105,MCM0,Mark C. Montigny,478
55,EDJ0,Edward J. Kennedy,478
24,CFF0,Cindy F. Friedman,478
77,JMC0,Joanne M. Comerford,478
132,PDJ0,Patricia D. Jehlen,478
117,MOM0,Michael O. Moore,478
1,AGH0,Adam G. Hinds,478
151,SCD0,Sonia Chang-Diaz,478
72,JFK0,John F. Keenan,478
16,BPC0,Brendan P. Crighton,478


In [ ]:
bad_codes = vote_counts[vote_counts["VoteCount"] <= 8]["MemberCode"].tolist()
bad_codes[:10]

['MJF1',
 'PKF1',
 'PJD2',
 'MJM1',
 'MJC1',
 'MJM2',
 'MDR1',
 'MJS3',
 'PJK1',
 'M_C1']

In [ ]:
bad_rows = merged_df_191[
    merged_df_191["MemberCode"].isin(bad_codes)
].copy()

print(bad_rows['RollCallNumber'].unique())
print(bad_rows['BillNumber'].unique())

[25 67 68 69 70 71 72 73]
['H86']


In [ ]:
merged_df_191_clean = merged_df_191[
    ~merged_df_191["MemberCode"].isin(bad_codes)
].copy()
merged_df_191_clean.head()

,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,191,DAT0,Dean A. Tran,Senate,S63,269,None,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
1,191,DAT0,Dean A. Tran,Senate,H1771,81,None,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
2,191,DAT0,Dean A. Tran,Senate,H86,25,None,Nay,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
3,191,DAT0,Dean A. Tran,Senate,H86,67,Question on adoption of the amendment #1,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...
4,191,DAT0,Dean A. Tran,Senate,H86,68,Question on adoption of the amendment #3,Yea,Republican,Worcester and Middlesex,Assistant Minority Whip,dean.tran@masenate.gov,(617) 722-1230,504,http://malegislature.gov/api/GeneralCourts/191...


In [ ]:
print(merged_df_191_clean['MemberCode'].unique())

['DAT0' 'MJB0' 'WNB0' 'HLC0' 'SCD0' 'CSC0' 'SND0' 'JBE0' 'PDJ0' 'JFK0'
 'JBL0' 'MCM0' 'MOM0' 'MRP0' 'MJR0' 'MFR0' 'BET0' 'JTW0' 'PMO' 'jml0'
 'MDB0' 'CFF0' 'RCF0' 'AMG0' 'EPL0' 'JAB0' 'WFT0' 'AGH0' 'JAC0' 'PRF0'
 'BPC0' 'N_C0' 'D_D0' 'JMC0' 'EDJ0' 'RLR0' 'BRF0' 'SLM0' 'JCV0' 'DFH0'
 'VMD0' 'KES0']


##### 190

In [ ]:
merged_df_190 = merge_legislator_votes(df_vote_details_190, df_legislators_190)
merged_df_190.head()

In [ ]:
merged_df_190['MemberCode'].unique()

##### 189

In [ ]:
merged_df_189 = merge_legislator_votes(df_vote_details_189, df_legislators_189)
merged_df_189.head()

##### 188

In [ ]:
merged_df_188 = merge_legislator_votes(df_vote_details_188, df_legislators_188)
merged_df_188.head()

In [ ]:
merged_df_188['MemberCode'].unique()

##### 186

In [ ]:
merged_df_186 = merge_legislator_votes(df_vote_details_186, df_legislators_186)
merged_df_186.head()

Loaded 1010 vote records and 40 legislators.
Merged 1010 rows.


,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,186,M_W0,Marian Walsh,Senate,H1696,293,None,Yea,None,Norfolk and Suffolk,None,None,None,None,http://malegislature.gov/api/GeneralCourts/186...
1,186,M_W0,Marian Walsh,Senate,H4156,348,None,Nay,None,Norfolk and Suffolk,None,None,None,None,http://malegislature.gov/api/GeneralCourts/186...
2,186,M_W0,Marian Walsh,Senate,H4316,295,None,Yea,None,Norfolk and Suffolk,None,None,None,None,http://malegislature.gov/api/GeneralCourts/186...
3,186,M_W0,Marian Walsh,Senate,H4368,291,None,Yea,None,Norfolk and Suffolk,None,None,None,None,http://malegislature.gov/api/GeneralCourts/186...
4,186,M_W0,Marian Walsh,Senate,H4470,190,None,Yea,None,Norfolk and Suffolk,None,None,None,None,http://malegislature.gov/api/GeneralCourts/186...


#### Save to CSV

In [40]:
merged_all = pd.concat([v for k, v in merged_dfs.items()], ignore_index=True).reset_index(drop=True)

In [41]:
print(merged_all.head())

   GeneralCourtNumber MemberCode               Name  Branch BillNumber  \
0                 186       SAB0  Steven A. Baddour  Senate      H1696   
1                 186       SAB0  Steven A. Baddour  Senate      H4156   
2                 186       SAB0  Steven A. Baddour  Senate      H4316   
3                 186       SAB0  Steven A. Baddour  Senate      H4368   
4                 186       SAB0  Steven A. Baddour  Senate      H4470   

   RollCallNumber QuestionMotion Vote Party     District LeadershipPosition  \
0             293           None  Yea  None  First Essex               None   
1             348           None  Yea  None  First Essex               None   
2             295           None  Yea  None  First Essex               None   
3             291           None  Yea  None  First Essex               None   
4             190           None  Yea  None  First Essex               None   

  EmailAddress PhoneNumber RoomNumber  \
0         None        None       None  

In [42]:
merged_all.to_csv("all_sessions_votes.csv", index=False, encoding="utf-8-sig")

In [ ]:
# df_all = pd.concat(
#     [merged_df_194, merged_df_193, 
#      merged_df_191_clean,merged_df_190,
#      merged_df_189,merged_df_188], 
#     ignore_index=True
# )


In [ ]:
# df_all.to_csv("all_sessions_votes_1.csv", index=False, encoding="utf-8-sig")

add 192 and 186

In [ ]:
# df_csv = pd.read_csv("all_sessions_votes_1.csv") 

In [ ]:
# df_clean = df_csv[df_csv["GeneralCourtNumber"] != 192].copy()

# print(df_clean["GeneralCourtNumber"].unique())
# print(df_clean.shape)
# 

[194 193 190 189 188]
(75989, 15)


In [ ]:
# df_all_session = pd.concat(
#     [df_clean, merged_df_192_clean, merged_df_191_clean, merged_df_186],
#     ignore_index=True
# )

In [ ]:
# df_all = pd.concat(
#     [df_csv, merged_df_192_clean, merged_df_186],
#     ignore_index=True
# )


In [ ]:
# df_all_session = df_all_session.sort_values(
#     by=["GeneralCourtNumber", "Name", "RollCallNumber"],
#     ascending=[False, True, True]
# ).reset_index(drop=True)
# df_all_session.head()

,GeneralCourtNumber,MemberCode,Name,Branch,BillNumber,RollCallNumber,QuestionMotion,Vote,Party,District,LeadershipPosition,EmailAddress,PhoneNumber,RoomNumber,MemberUrl
0,194,A_G0,Adam Gómez,Senate,S15,2,Question on adoption of the amendment 2,Nay,Democrat,Hampden,NaN,Adam.Gomez@masenate.gov,(617) 722-1660,312-E,http://malegislature.gov/api/GeneralCourts/194...
1,194,A_G0,Adam Gómez,Senate,S15,3,Question on adoption of the amendment 9,Nay,Democrat,Hampden,NaN,Adam.Gomez@masenate.gov,(617) 722-1660,312-E,http://malegislature.gov/api/GeneralCourts/194...
2,194,A_G0,Adam Gómez,Senate,S15,4,Question on adoption of the amendment 10,Nay,Democrat,Hampden,NaN,Adam.Gomez@masenate.gov,(617) 722-1660,312-E,http://malegislature.gov/api/GeneralCourts/194...
3,194,A_G0,Adam Gómez,Senate,S15,5,Question on adoption of the amendment 13,Nay,Democrat,Hampden,NaN,Adam.Gomez@masenate.gov,(617) 722-1660,312-E,http://malegislature.gov/api/GeneralCourts/194...
4,194,A_G0,Adam Gómez,Senate,S15,6,Question on adoption of the amendment 22,Nay,Democrat,Hampden,NaN,Adam.Gomez@masenate.gov,(617) 722-1660,312-E,http://malegislature.gov/api/GeneralCourts/194...


In [ ]:
# print(df_all_session["GeneralCourtNumber"].unique())

[194 193 192 191 190 189 188 186]


In [ ]:
# df_all_session.to_csv("all_sessions_senate_votes.csv", index=False, encoding="utf-8-sig")